# Notebook 07 — Pipeline RAG Completo

**Semana:** 7  
**Bloque:** 4.4 — Pipeline retrieval + generación  
**Tiempo estimado:** 3.5-4 horas

## Qué vas a construir

Al terminar este notebook tendrás un chatbot RAG funcional sobre cualquier corpus de documentos:
- Pipeline de ingesta modular (cargar → limpiar → chunkear → indexar)
- Pipeline de consulta completo (embed → retrieve → prompt → generar)
- Query rewriting y multi-query retrieval
- Manejo del caso "no hay información relevante"
- **Reto final:** chatbot RAG sobre la documentación de FastAPI con historial de conversación

## Prerrequisitos

Semanas 5 y 6 completadas. Chroma y las funciones de embedding disponibles.

```bash
pip install chromadb langchain-text-splitters openai anthropic pymupdf requests python-dotenv
```

---
## Celda 0 — Setup: imports y clientes

In [ ]:
import os
import re
import json
import hashlib
import datetime
import requests
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import Literal
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

cliente_openai    = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
cliente_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
chroma_cliente    = chromadb.PersistentClient(path="./chroma_db")

MODELO_EMBEDDING  = "text-embedding-3-small"
MODELO_GENERACION = "gpt-4o-mini"

# ── Helpers de embedding (de semanas anteriores) ─────────────────────────────
def generar_embedding(texto: str) -> list[float]:
    r = cliente_openai.embeddings.create(model=MODELO_EMBEDDING, input=texto)
    return r.data[0].embedding

def generar_embeddings_batch(textos: list[str]) -> np.ndarray:
    r = cliente_openai.embeddings.create(model=MODELO_EMBEDDING, input=textos)
    return np.array([item.embedding for item in sorted(r.data, key=lambda x: x.index)])

# ── Helper de generación ──────────────────────────────────────────────────────
def completar(
    mensajes: list[dict],
    system: str | None = None,
    temperature: float = 0.1,
    max_tokens: int = 1024
) -> str:
    """Llamada simple al LLM, devuelve el texto."""
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente_openai.chat.completions.create(
        model=MODELO_GENERACION,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return r.choices[0].message.content

Path("datos_rag").mkdir(exist_ok=True)
print("Setup completado.")

---
## Celda 1 — Pipeline de ingesta modular

Construye cada etapa como una función independiente.
Al final encadénalas en `ingestar_documentos()`.

Usamos la documentación de FastAPI (descargada desde GitHub en Markdown)
porque es un corpus real, bien estructurado y en inglés/español.

In [ ]:
# ── Fuentes de documentos ─────────────────────────────────────────────────────
# Descargamos páginas de la doc de FastAPI en texto plano desde Wikipedia
# (alternativa gratuita sin necesidad de scraping complejo)

FUENTES = [
    {"titulo": "FastAPI",                  "tipo": "framework"},
    {"titulo": "Representational state transfer", "tipo": "concepto"},
    {"titulo": "Pydantic",                 "tipo": "libreria"},
    {"titulo": "JSON Web Token",           "tipo": "seguridad"},
    {"titulo": "Docker (software)",        "tipo": "infraestructura"},
    {"titulo": "PostgreSQL",               "tipo": "base_datos"},
]

# ── Etapa 1: Loader — carga texto desde Wikipedia ────────────────────────────

def cargar_wikipedia(titulo: str) -> str:
    """Descarga el texto completo de un artículo de Wikipedia en inglés."""
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query", "format": "json",
        "titles": titulo, "prop": "extracts",
        "explaintext": True, "exsectionformat": "plain"
    }
    try:
        resp   = requests.get(url, params=params, timeout=10)
        pages  = resp.json()["query"]["pages"]
        texto  = list(pages.values())[0].get("extract", "")
        return texto
    except Exception as e:
        print(f"  Error cargando '{titulo}': {e}")
        return ""

# ── Etapa 2: Cleaner — limpieza básica ───────────────────────────────────────

def limpiar(texto: str) -> str:
    """Elimina ruido: caracteres de control, espacios múltiples, secciones vacías."""
    texto = re.sub(r"[\x00-\x08\x0b\x0e-\x1f\x7f]", "", texto)
    texto = re.sub(r" +",    " ",   texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = re.sub(r"== References ==.*", "", texto, flags=re.DOTALL)  # quita refs Wikipedia
    return texto.strip()

# ── Etapa 3: Chunker ─────────────────────────────────────────────────────────

_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

def chunkear(texto: str) -> list[str]:
    """Divide el texto en chunks de 500 caracteres con 100 de solapamiento."""
    return _splitter.split_text(texto)

# ── Etapa 4: Indexer — guarda en Chroma ──────────────────────────────────────

def indexar_en_coleccion(
    coleccion: chromadb.Collection,
    chunks: list[str],
    metadatos_base: dict,
    batch_size: int = 50
) -> int:
    """Embeddea e indexa chunks. Devuelve el número de chunks añadidos."""
    total = len(chunks)
    for inicio in range(0, total, batch_size):
        lote      = chunks[inicio:min(inicio + batch_size, total)]
        ids       = [
            f"{metadatos_base['fuente']}_{i}"
            for i in range(inicio, inicio + len(lote))
        ]
        embeddings = generar_embeddings_batch(lote).tolist()
        metas      = [
            {**metadatos_base, "chunk_index": i, "total_chunks": total, "n_chars": len(c)}
            for i, c in enumerate(lote, inicio)
        ]
        coleccion.upsert(ids=ids, documents=lote, embeddings=embeddings, metadatas=metas)
    return total

# ── Pipeline de ingesta completo ─────────────────────────────────────────────

def ingestar_documentos(
    fuentes: list[dict],
    nombre_coleccion: str = "rag_docs"
) -> chromadb.Collection:
    """Orquesta Loader → Cleaner → Chunker → Indexer para todas las fuentes."""
    try:
        chroma_cliente.delete_collection(nombre_coleccion)
    except Exception:
        pass
    coleccion = chroma_cliente.create_collection(
        name=nombre_coleccion,
        metadata={"hnsw:space": "cosine"}
    )

    total_chunks = 0
    for fuente in fuentes:
        titulo = fuente["titulo"]
        print(f"\nIngesta: '{titulo}'")

        texto    = cargar_wikipedia(titulo)
        if not texto:
            print(f"  Saltando (sin contenido)")
            continue

        texto    = limpiar(texto)
        chunks   = chunkear(texto)
        n        = indexar_en_coleccion(
            coleccion, chunks,
            metadatos_base={
                "fuente":        titulo,
                "tipo":          fuente.get("tipo", "general"),
                "fecha_ingesta": datetime.date.today().isoformat(),
                "idioma":        "en"
            }
        )
        print(f"  → {n} chunks indexados")
        total_chunks += n

    print(f"\nIngesta completada. Total: {total_chunks} chunks en '{nombre_coleccion}'")
    return coleccion


# ── Ejecuta la ingesta ────────────────────────────────────────────────────────
coleccion_rag = ingestar_documentos(FUENTES, nombre_coleccion="rag_docs")
print(f"\nDocumentos en la colección: {coleccion_rag.count()}")

---
## Celda 2 — Pipeline de consulta básico

Construye las tres funciones del pipeline de consulta:
`recuperar()`, `construir_prompt()`, `generar()`.
Luego encadénalas en `responder()` y prueba con varias preguntas.

In [ ]:
UMBRAL_RELEVANCIA = 0.68

SYSTEM_RAG = """You are a technical documentation assistant.
Answer questions using ONLY the information provided in the context.
If the answer is not found in the context, say: 'I don't have enough information in the available documentation to answer this.'
Do not use external knowledge. Always cite the source at the end of your answer."""

# ── Función 1: recuperar ─────────────────────────────────────────────────────

def recuperar(
    query: str,
    coleccion: chromadb.Collection,
    k: int = 4,
    filtro: dict | None = None
) -> list[dict]:
    """
    Embeddea la query y recupera los k chunks más similares.
    Devuelve lista de dicts con texto, similitud y metadatos.
    Nota: Chroma con métrica coseno devuelve distancia (0=idéntico, 2=opuesto).
          Convertimos a similitud: sim = 1 - distancia.
    """
    vector = generar_embedding(query)
    kwargs = {
        "query_embeddings": [vector],
        "n_results":        min(k, coleccion.count()),
        "include":          ["documents", "distances", "metadatas"]
    }
    if filtro:
        kwargs["where"] = filtro

    resp = coleccion.query(**kwargs)

    resultados = []
    for doc, dist, meta in zip(
        resp["documents"][0],
        resp["distances"][0],
        resp["metadatas"][0]
    ):
        resultados.append({
            "texto":     doc,
            "similitud": round(1 - dist, 4),
            "meta":      meta
        })
    return resultados


# ── Función 2: construir_prompt ──────────────────────────────────────────────

def construir_prompt(query: str, chunks: list[dict]) -> str:
    """
    Formatea los chunks recuperados como contexto numerado.
    Incluye score de relevancia y metadatos para que el LLM pueda citar.
    """
    bloques = []
    for i, chunk in enumerate(chunks, 1):
        meta  = chunk["meta"]
        fuente = meta.get("fuente", "desconocida")
        score  = chunk["similitud"]
        bloques.append(
            f"[Context {i} | Source: {fuente} | Relevance: {score:.2f}]\n"
            f"{chunk['texto']}"
        )

    contexto = "\n\n---\n\n".join(bloques)

    return f"""CONTEXT:
{contexto}

QUESTION: {query}

ANSWER (based only on the context above):"""


# ── Función 3: orquestador ───────────────────────────────────────────────────

@dataclass
class RespuestaRAG:
    pregunta:    str
    respuesta:   str
    fuentes:     list[dict]
    n_chunks:    int
    respondida:  bool    # False si no hubo chunks suficientemente relevantes

    def mostrar(self) -> None:
        estado = "✓" if self.respondida else "⚠ Sin información suficiente"
        print(f"\n{estado}")
        print(f"Pregunta:  {self.pregunta}")
        print(f"Respuesta: {self.respuesta}")
        if self.fuentes:
            fuentes_unicas = list({f.get("fuente", "?") for f in self.fuentes})
            print(f"Fuentes ({self.n_chunks} chunks): {', '.join(fuentes_unicas)}")


def responder(
    pregunta: str,
    coleccion: chromadb.Collection,
    k: int = 4,
    umbral: float = UMBRAL_RELEVANCIA,
    filtro: dict | None = None
) -> RespuestaRAG:
    """Pipeline completo: retrieve → filtrar por umbral → construir prompt → generar."""

    # Recupera y filtra por relevancia
    chunks_todos    = recuperar(pregunta, coleccion, k=k, filtro=filtro)
    chunks_relevantes = [c for c in chunks_todos if c["similitud"] >= umbral]

    if not chunks_relevantes:
        return RespuestaRAG(
            pregunta=pregunta,
            respuesta="I don't have enough information in the available documentation to answer this.",
            fuentes=[],
            n_chunks=0,
            respondida=False
        )

    prompt    = construir_prompt(pregunta, chunks_relevantes)
    respuesta = completar(
        mensajes=[{"role": "user", "content": prompt}],
        system=SYSTEM_RAG
    )

    return RespuestaRAG(
        pregunta=pregunta,
        respuesta=respuesta,
        fuentes=[c["meta"] for c in chunks_relevantes],
        n_chunks=len(chunks_relevantes),
        respondida=True
    )


# ── Pruebas del pipeline básico ───────────────────────────────────────────────
PREGUNTAS_TEST = [
    "What is FastAPI and what are its main features?",
    "How does JWT authentication work?",
    "What is Docker and why is it used for deployment?",
    "What is the meaning of life?",   # pregunta fuera del corpus — debe fallar bien
]

for p in PREGUNTAS_TEST:
    r = responder(p, coleccion_rag)
    r.mostrar()
    print()

**Observaciones** (edita esta celda):

- ¿Las respuestas citan correctamente la fuente?
- ¿La pregunta fuera de corpus devolvió el mensaje de "sin información"?
- ¿Alguna respuesta contiene información que NO está en el contexto?

---
## Celda 3 — Query rewriting y multi-query retrieval

Dos técnicas que mejoran el retrieval antes de llamar al LLM.
Compara los chunks que recupera con y sin cada técnica.

In [ ]:
# ── Técnica 1: query rewriting ────────────────────────────────────────────────

def reescribir_query(pregunta: str) -> str:
    """
    Reformula la pregunta del usuario para optimizarla para búsqueda semántica.
    Útil cuando el usuario usa lenguaje coloquial o ambiguo.
    """
    prompt = f"""Rewrite the following question as an optimized semantic search query.
The rewritten query should:
- Be more specific and technical
- Remove conversational filler words
- Use key technical terms
- Be in declarative form if more effective

Original question: {pregunta}
Optimized query (only the query, no explanation):"""

    return completar(
        mensajes=[{"role": "user", "content": prompt}],
        temperature=0.0
    ).strip()


# ── Técnica 2: multi-query retrieval ─────────────────────────────────────────

def generar_queries_alternativas(pregunta: str, n: int = 3) -> list[str]:
    """
    Genera N reformulaciones distintas de la misma pregunta.
    Cada una aborda el concepto desde un ángulo diferente.
    """
    prompt = f"""Generate {n} different search queries to find information about the same topic as this question.
Each query should approach the concept from a different angle.
Return only the queries, one per line, no numbering or explanation.

Question: {pregunta}"""

    respuesta = completar(
        mensajes=[{"role": "user", "content": prompt}],
        temperature=0.4
    )
    return [q.strip() for q in respuesta.strip().split("\n") if q.strip()][:n]


def recuperar_multi_query(
    pregunta: str,
    coleccion: chromadb.Collection,
    k_por_query: int = 3,
    n_queries: int = 3
) -> list[dict]:
    """
    Genera N queries alternativas, recupera con cada una y deduplica.
    Devuelve los chunks únicos ordenados por máxima similitud.
    """
    queries_alt = generar_queries_alternativas(pregunta, n=n_queries)
    todas_queries = [pregunta] + queries_alt

    print(f"  Queries generadas:")
    for q in todas_queries:
        print(f"    - {q}")

    # Recupera con todas las queries
    vistos   = set()   # para deduplicar por texto
    unicos   = []
    for q in todas_queries:
        chunks = recuperar(q, coleccion, k=k_por_query)
        for c in chunks:
            clave = c["texto"][:100]   # fingerprint del chunk
            if clave not in vistos:
                vistos.add(clave)
                unicos.append(c)

    # Ordena por similitud descendente y toma los top-k globales
    unicos.sort(key=lambda x: -x["similitud"])
    return unicos[:k_por_query * 2]   # top 2×k chunks únicos


# ── Comparativa: básico vs rewriting vs multi-query ───────────────────────────

QUERY_COLOQUIAL = "hey, how do you keep APIs safe from bad actors?"

print(f"Query original: '{QUERY_COLOQUIAL}'")
print("=" * 60)

# Básico: query tal cual
print("\n[1] Pipeline BÁSICO (sin rewriting):")
chunks_basico = recuperar(QUERY_COLOQUIAL, coleccion_rag, k=3)
for c in chunks_basico:
    print(f"  {c['similitud']:.4f} | [{c['meta']['fuente'][:20]}] {c['texto'][:70]}...")

# Con rewriting
print("\n[2] Con QUERY REWRITING:")
query_reescrita = reescribir_query(QUERY_COLOQUIAL)
print(f"  Query reescrita: '{query_reescrita}'")
chunks_rewrite = recuperar(query_reescrita, coleccion_rag, k=3)
for c in chunks_rewrite:
    print(f"  {c['similitud']:.4f} | [{c['meta']['fuente'][:20]}] {c['texto'][:70]}...")

# Multi-query
print("\n[3] Con MULTI-QUERY RETRIEVAL:")
chunks_multi = recuperar_multi_query(QUERY_COLOQUIAL, coleccion_rag)
print(f"  Total chunks únicos recuperados: {len(chunks_multi)}")
for c in chunks_multi[:4]:
    print(f"  {c['similitud']:.4f} | [{c['meta']['fuente'][:20]}] {c['texto'][:70]}...")

**Observaciones** (edita esta celda):

- ¿La query reescrita mejoró los scores de similitud?
- ¿El multi-query recuperó chunks que el básico no encontró? ¿Cuáles?
- ¿Cuándo vale la pena el coste extra de multi-query (2 llamadas al LLM extra)?

---
## Celda 4 — Responder con multi-query y verificar no-respuesta

Conecta el multi-query al pipeline completo y prueba el comportamiento
ante preguntas fuera del corpus.

In [ ]:
def responder_avanzado(
    pregunta: str,
    coleccion: chromadb.Collection,
    usar_rewriting: bool = True,
    usar_multi_query: bool = False,
    k: int = 4,
    umbral: float = UMBRAL_RELEVANCIA,
    verbose: bool = False
) -> RespuestaRAG:
    """
    Pipeline avanzado con rewriting y/o multi-query opcionales.
    """
    query_efectiva = pregunta

    # Paso opcional: reescribir la query
    if usar_rewriting:
        query_efectiva = reescribir_query(pregunta)
        if verbose:
            print(f"  Query reescrita: '{query_efectiva}'")

    # Recuperación
    if usar_multi_query:
        chunks_todos = recuperar_multi_query(query_efectiva, coleccion, k_por_query=k)
    else:
        chunks_todos = recuperar(query_efectiva, coleccion, k=k)

    chunks_relevantes = [c for c in chunks_todos if c["similitud"] >= umbral]

    if verbose:
        print(f"  Chunks recuperados: {len(chunks_todos)}, relevantes (>={umbral}): {len(chunks_relevantes)}")

    if not chunks_relevantes:
        return RespuestaRAG(
            pregunta=pregunta,
            respuesta="I don't have enough information in the available documentation to answer this.",
            fuentes=[], n_chunks=0, respondida=False
        )

    prompt    = construir_prompt(pregunta, chunks_relevantes)
    respuesta = completar(
        mensajes=[{"role": "user", "content": prompt}],
        system=SYSTEM_RAG
    )

    return RespuestaRAG(
        pregunta=pregunta,
        respuesta=respuesta,
        fuentes=[c["meta"] for c in chunks_relevantes],
        n_chunks=len(chunks_relevantes),
        respondida=True
    )


# ── Test del caso de no-respuesta: preguntas fuera del corpus ─────────────────

print("=" * 60)
print("TEST: PREGUNTAS FUERA DEL CORPUS")
print("=" * 60)

PREGUNTAS_FUERA = [
    "What is the current price of Bitcoin?",
    "Who won the last FIFA World Cup?",
    "What is the best pizza recipe?",
]

for p in PREGUNTAS_FUERA:
    r = responder_avanzado(p, coleccion_rag, verbose=True)
    icono = "✓ Correctamente rechazada" if not r.respondida else "✗ ERROR: respondió sin información"
    print(f"\n{icono}: '{p}'")
    if r.respondida:
        print(f"  Respuesta (no debería existir): {r.respuesta[:100]}...")

# ── Comparativa rewriting vs sin rewriting en pregunta real ───────────────────

print("\n" + "=" * 60)
print("COMPARATIVA: con y sin rewriting")
print("=" * 60)

PREGUNTA_COLOQUIAL = "I heard Pydantic is useful, what does it do exactly?"

r_sin = responder(PREGUNTA_COLOQUIAL, coleccion_rag, k=3)
r_con = responder_avanzado(PREGUNTA_COLOQUIAL, coleccion_rag, usar_rewriting=True, k=3, verbose=True)

print(f"\nRespuesta SIN rewriting ({r_sin.n_chunks} chunks):")
print(f"  {r_sin.respuesta[:200]}...")
print(f"\nRespuesta CON rewriting ({r_con.n_chunks} chunks):")
print(f"  {r_con.respuesta[:200]}...")

---
## Reto final — Chatbot RAG con historial de conversación

Extiende el pipeline para que el asistente recuerde la conversación anterior.
El historial permite hacer preguntas de seguimiento: "¿y para qué sirve eso?"
o "dame más detalles sobre el tercer punto".

Lee el esqueleto y completa los `# TODO`.

In [ ]:
@dataclass
class ChatRAG:
    """
    Chatbot RAG con historial de conversación.

    El historial sirve para dos cosas:
    1. Contexto conversacional al LLM (recuerda turnos anteriores)
    2. Enriquecer la query de retrieval con el contexto previo
    """
    coleccion:       chromadb.Collection
    k:               int   = 4
    umbral:          float = 0.65
    max_historial:   int   = 6    # máximo de turnos a mantener (para controlar tokens)
    historial:       list  = field(default_factory=list)
    fuentes_sesion:  list  = field(default_factory=list)

    def _construir_query_con_contexto(self, pregunta: str) -> str:
        """
        Enriquece la query con el contexto de los últimos turnos.
        Útil para preguntas de seguimiento como "¿y para qué sirve eso?".
        """
        if not self.historial:
            return pregunta

        # Toma solo los últimos 2 turnos para no sobrecargar
        contexto_reciente = self.historial[-4:]   # 2 turnos = 4 mensajes (user+assistant)
        resumen_contexto  = " ".join(
            m["content"][:100] for m in contexto_reciente if m["role"] == "assistant"
        )

        if len(resumen_contexto) < 20:   # no hay contexto útil
            return pregunta

        return f"{pregunta} (context: {resumen_contexto[:200]})"

    def responder(self, pregunta: str, verbose: bool = False) -> str:
        """
        Responde una pregunta usando RAG + historial de conversación.

        Flujo:
        1. Enriquece la query con el historial para mejorar el retrieval
        2. Recupera chunks relevantes
        3. Construye el prompt incluyendo el historial de conversación
        4. Genera la respuesta
        5. Añade el turno al historial
        """
        # TODO: implementa el flujo completo
        #
        # Paso 1: enriquece la query
        # query_retrieval = self._construir_query_con_contexto(pregunta)
        #
        # Paso 2: recupera chunks
        # chunks = recuperar(query_retrieval, self.coleccion, k=self.k)
        # chunks_relevantes = [c for c in chunks if c["similitud"] >= self.umbral]
        #
        # Paso 3: si no hay chunks relevantes, devuelve mensaje estándar
        #
        # Paso 4: construye el prompt con contexto RAG
        # (usa construir_prompt() de la celda anterior)
        #
        # Paso 5: llama al LLM con el historial de conversación
        # Tip: los mensajes enviados al LLM deben ser:
        #   [system] + historial[-max_historial:] + [user: prompt_con_contexto]
        # NO incluyas el historial en el retrieval prompt,
        # solo en la llamada al LLM.
        #
        # Paso 6: guarda en el historial
        # self.historial.append({"role": "user",      "content": pregunta})
        # self.historial.append({"role": "assistant", "content": texto_respuesta})
        # self.fuentes_sesion.extend([c["meta"] for c in chunks_relevantes])

        raise NotImplementedError("Implementa el método responder()")

    def mostrar_historial(self) -> None:
        """Muestra el historial de la conversación."""
        print(f"\n{'='*60}")
        print(f"Historial ({len(self.historial)//2} turnos)")
        print(f"{'='*60}")
        for m in self.historial:
            rol    = "Tú" if m["role"] == "user" else "Asistente"
            print(f"\n[{rol}]")
            print(f"  {m['content'][:200]}{'...' if len(m['content']) > 200 else ''}")

    def resetear(self) -> None:
        """Limpia el historial para empezar una nueva sesión."""
        self.historial      = []
        self.fuentes_sesion = []
        print("Historial reseteado.")


# ── Prueba del chatbot con preguntas de seguimiento ───────────────────────────
# Descomenta cuando hayas implementado ChatRAG.responder()

# chat = ChatRAG(coleccion=coleccion_rag, k=4, umbral=0.65)

# DIALOGO = [
#     "What is FastAPI?",
#     "What are its main advantages over Flask?",   # pregunta de seguimiento
#     "How does it handle data validation?",         # sigue hablando de FastAPI
#     "And what library does it use for that?",       # referencia implícita a Pydantic
#     "What is the best programming language?",       # fuera del corpus — debe fallar bien
# ]

# for turno, pregunta in enumerate(DIALOGO, 1):
#     print(f"\n[Turno {turno}] Pregunta: {pregunta}")
#     print("-" * 50)
#     respuesta = chat.responder(pregunta, verbose=True)
#     print(f"Respuesta: {respuesta[:300]}..." if len(respuesta) > 300 else f"Respuesta: {respuesta}")

# chat.mostrar_historial()

### Extensiones opcionales

1. **Streaming en el reto:** modifica `ChatRAG.responder()` para que use streaming y muestre los tokens en tiempo real usando `completar_stream()` de la semana 3.

2. **Compresión del historial:** cuando el historial supera `max_historial` turnos, en lugar de descartarlos, pide al LLM que los resuma en 2-3 frases y usa ese resumen como contexto.

3. **Interfaz Streamlit:** construye una interfaz web mínima con Streamlit que envuelva `ChatRAG`. Será la base del Proyecto 1 de la semana 12.

In [ ]:
# Extensión 3: esqueleto de interfaz Streamlit
# Guarda esto como app_rag.py y ejecútalo con: streamlit run app_rag.py

STREAMLIT_APP = '''
import streamlit as st
import os
import chromadb
from dotenv import load_dotenv

# Importa aquí tus funciones del notebook
# from rag_pipeline import ChatRAG, ingestar_documentos, ...

load_dotenv()

st.set_page_config(page_title="RAG Chatbot", page_icon="🤖")
st.title("📚 RAG Chatbot")
st.caption("Pregunta sobre la documentación indexada")

# Estado de sesión
if "mensajes" not in st.session_state:
    st.session_state.mensajes = []
if "chat" not in st.session_state:
    chroma_cli = chromadb.PersistentClient(path="./chroma_db")
    coleccion  = chroma_cli.get_collection("rag_docs")
    # st.session_state.chat = ChatRAG(coleccion=coleccion)

# Muestra el historial
for msg in st.session_state.mensajes:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Input del usuario
if prompt := st.chat_input("Escribe tu pregunta..."):
    st.session_state.mensajes.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Buscando en la documentación..."):
            # respuesta = st.session_state.chat.responder(prompt)
            respuesta = "[Implementa ChatRAG primero]"
        st.markdown(respuesta)

    st.session_state.mensajes.append({"role": "assistant", "content": respuesta})
'''

with open("app_rag.py", "w") as f:
    f.write(STREAMLIT_APP)

print("app_rag.py creado. Para ejecutarlo:")
print("  pip install streamlit")
print("  streamlit run app_rag.py")

---
## Autoevaluación de la semana

1. **¿Qué funcionó bien?**  
   ...

2. **¿Qué no entendiste del todo?**  
   ...

3. **¿Qué cambiarías la semana que viene?**  
   ...

4. **Tiempo real dedicado:**  
   ...

---
*Notebook 07 — Semana 7 del Plan de Formación AI Engineering*